# 04_make_grouped_splits

This notebook creates **lesion-grouped splits** for the current primary experiment:

- **Classes:** 6-class
  - `NV`, `MEL`, `BCC`, `BKL`, `AKIEC`, `DF`
- **Base training source:** `HAM10000`
- **Target adaptation / evaluation source:** `BCN20000`

## Output files

Saved into `../data/processed/splits/`:

- `train_base.csv`
- `val_base.csv`
- `train_bcn_ft.csv`
- `val_bcn_ft.csv`
- `test_bcn.csv`

## Why grouped splits?

A lesion can have multiple images. If different images from the same lesion appear in different splits, the model can leak information from train to val/test.

So we split by **`lesion_id`**, not by image.


## Notebook plan

1. Load cleaned HAM and BCN metadata
2. Keep only rows for the primary experiment
3. Confirm the 6-class taxonomy
4. Split by `lesion_id`
5. Run leakage checks
6. Print class and lesion distributions
7. Save final split CSVs


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

SEED = 42

HAM_CSV = Path("../data/processed/ham10000_clean_metadata.csv")
BCN_CSV = Path("../data/processed/bcn20000_clean_metadata.csv")
OUT_DIR = Path("../data/processed/splits")

OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_CLASSES = ["NV", "MEL", "BCC", "BKL", "AKIEC", "DF"]

print("HAM_CSV exists:", HAM_CSV.exists(), HAM_CSV)
print("BCN_CSV exists:", BCN_CSV.exists(), BCN_CSV)
print("OUT_DIR:", OUT_DIR.resolve())
print("SEED:", SEED)
print("PRIMARY_CLASSES:", PRIMARY_CLASSES)


HAM_CSV exists: True ../data/processed/ham10000_clean_metadata.csv
BCN_CSV exists: True ../data/processed/bcn20000_clean_metadata.csv
OUT_DIR: /Users/choekyelnyungmartsang/Developer/master-thesis/data/processed/splits
SEED: 42
PRIMARY_CLASSES: ['NV', 'MEL', 'BCC', 'BKL', 'AKIEC', 'DF']


## Load cleaned metadata

These CSVs should come from your previous cleaning notebook.


In [2]:
ham_df = pd.read_csv(HAM_CSV)
bcn_df = pd.read_csv(BCN_CSV)

print("Loaded HAM:", ham_df.shape)
print("Loaded BCN:", bcn_df.shape)

print("\nHAM columns:")
print(list(ham_df.columns))

print("\nBCN columns:")
print(list(bcn_df.columns))


Loaded HAM: (11720, 32)
Loaded BCN: (18946, 32)

HAM columns:
['isic_id', 'source_dataset', 'image_path', 'image_exists', 'lesion_id', 'group_id', 'raw_label', 'harmonized_label', 'primary_status', 'keep_for_primary_experiment', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'melanocytic', 'sex', 'sex_clean', 'age_approx', 'anatom_site_general', 'attribution', 'copyright_license', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_special', 'concomitant_biopsy', 'image_manipulation', 'raw_label_norm', 'sha256', 'exact_duplicate_group_size', 'has_exact_duplicate']

BCN columns:
['isic_id', 'source_dataset', 'image_path', 'image_exists', 'lesion_id', 'group_id', 'raw_label', 'harmonized_label', 'primary_status', 'keep_for_primary_experiment', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'melanocytic', 'sex', 'sex_clean', 'age_approx', 'anatom_site_general', 'attribution', 'copyright_license', 'anat

## Small helpers

We use a few helper functions to:
- coerce keep flags
- print distributions
- perform grouped splits
- run leakage checks


In [3]:
def to_bool_series(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }).fillna(False)

def print_split_summary(df, name, class_order=PRIMARY_CLASSES):
    print("\n" + "=" * 80)
    print(f"{name} summary")
    print("=" * 80)
    print("rows:", len(df))
    print("unique lesions:", df["lesion_id"].nunique(dropna=True))
    print("unique images:", df["isic_id"].nunique(dropna=True))

    print("\nClass counts:")
    counts = (
        df["harmonized_label"]
        .value_counts(dropna=False)
        .reindex(class_order, fill_value=0)
        .to_frame("count")
    )
    display(counts)

    print("\nClass proportions:")
    props = (
        df["harmonized_label"]
        .value_counts(normalize=True, dropna=False)
        .reindex(class_order, fill_value=0.0)
        .mul(100)
        .round(2)
        .to_frame("percent")
    )
    display(props)

def lesion_level_label_summary(df, name):
    tmp = (
        df.groupby("lesion_id")
        .agg(
            n_images=("isic_id", "size"),
            n_labels=("harmonized_label", lambda x: x.nunique(dropna=True)),
            labels=("harmonized_label", lambda x: sorted(pd.Series(x).dropna().unique()))
        )
        .reset_index()
    )
    conflicts = tmp[tmp["n_labels"] > 1].copy()

    print("\n" + "=" * 80)
    print(f"{name}: lesion-level label summary")
    print("=" * 80)
    print("unique lesions:", len(tmp))
    print("lesions with >1 label:", len(conflicts))

    if len(conflicts) > 0:
        print("\nTop lesion-label conflicts:")
        display(conflicts.sort_values(["n_labels", "n_images"], ascending=False).head(20))
    else:
        print("No lesion-level label conflicts found.")

    return tmp, conflicts

def grouped_split_by_lesion(
    df,
    group_col="lesion_id",
    label_col="harmonized_label",
    train_frac=0.8,
    val_frac=0.2,
    seed=42
):
    assert abs(train_frac + val_frac - 1.0) < 1e-8, "train_frac + val_frac must equal 1"

    rng = np.random.default_rng(seed)

    lesion_table = (
        df.groupby(group_col)
        .agg(
            n_images=("isic_id", "size"),
            label=(label_col, lambda x: pd.Series(x).dropna().iloc[0])
        )
        .reset_index()
    )

    train_lesions = []
    val_lesions = []

    for label, g in lesion_table.groupby("label"):
        lesions = g[group_col].tolist()
        lesions = list(rng.permutation(lesions))

        n = len(lesions)
        n_train = int(round(n * train_frac))
        n_train = max(1, min(n_train, n - 1)) if n > 1 else n

        train_lesions.extend(lesions[:n_train])
        val_lesions.extend(lesions[n_train:])

    train_df = df[df[group_col].isin(train_lesions)].copy()
    val_df = df[df[group_col].isin(val_lesions)].copy()

    return train_df, val_df

def grouped_split_train_val_test_by_lesion(
    df,
    group_col="lesion_id",
    label_col="harmonized_label",
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=42
):
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-8, "fractions must sum to 1"

    rng = np.random.default_rng(seed)

    lesion_table = (
        df.groupby(group_col)
        .agg(
            n_images=("isic_id", "size"),
            label=(label_col, lambda x: pd.Series(x).dropna().iloc[0])
        )
        .reset_index()
    )

    train_lesions = []
    val_lesions = []
    test_lesions = []

    for label, g in lesion_table.groupby("label"):
        lesions = g[group_col].tolist()
        lesions = list(rng.permutation(lesions))

        n = len(lesions)

        if n == 1:
            train_lesions.extend(lesions)
            continue
        if n == 2:
            train_lesions.append(lesions[0])
            test_lesions.append(lesions[1])
            continue

        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))

        n_train = max(1, n_train)
        n_val = max(1, n_val)
        n_test = n - n_train - n_val

        if n_test < 1:
            n_test = 1
            if n_train >= n_val and n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1

        while n_train + n_val + n_test > n:
            if n_train >= n_val and n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1
            else:
                n_test -= 1

        train_lesions.extend(lesions[:n_train])
        val_lesions.extend(lesions[n_train:n_train+n_val])
        test_lesions.extend(lesions[n_train+n_val:n_train+n_val+n_test])

    train_df = df[df[group_col].isin(train_lesions)].copy()
    val_df = df[df[group_col].isin(val_lesions)].copy()
    test_df = df[df[group_col].isin(test_lesions)].copy()

    return train_df, val_df, test_df

def check_no_overlap(df_a, df_b, col, name_a, name_b):
    a = set(df_a[col].dropna().unique())
    b = set(df_b[col].dropna().unique())
    overlap = a & b
    print(f"Overlap check for '{col}' between {name_a} and {name_b}: {len(overlap)}")
    if len(overlap) > 0:
        print("Preview overlaps:", sorted(list(overlap))[:20])
    return overlap

def save_split(df, path):
    df = df.copy()
    df.to_csv(path, index=False)
    print(f"Saved: {path} | shape={df.shape}")


## Keep only primary-experiment rows

We only keep rows that:
- passed label harmonization
- are marked for the primary experiment
- belong to the selected 6 classes


In [4]:
for df in [ham_df, bcn_df]:
    df["keep_for_primary_experiment"] = to_bool_series(df["keep_for_primary_experiment"])

ham_keep = ham_df.loc[
    ham_df["keep_for_primary_experiment"]
    & ham_df["harmonized_label"].isin(PRIMARY_CLASSES)
].copy()

bcn_keep = bcn_df.loc[
    bcn_df["keep_for_primary_experiment"]
    & bcn_df["harmonized_label"].isin(PRIMARY_CLASSES)
].copy()

print_split_summary(ham_keep, "HAM keep-only")
print_split_summary(bcn_keep, "BCN keep-only")



HAM keep-only summary
rows: 11310
unique lesions: 8542
unique images: 11310

Class counts:


,count
harmonized_label,
NV,7736
MEL,1305
BCC,622
BKL,1338
AKIEC,149
DF,160



Class proportions:


,percent
harmonized_label,
NV,68.40
MEL,11.54
BCC,5.50
BKL,11.83
AKIEC,1.32
DF,1.41



BCN keep-only summary
rows: 16133
unique lesions: 4543
unique images: 16133

Class counts:


,count
harmonized_label,
NV,5647
MEL,4003
BCC,3676
BKL,1551
AKIEC,1088
DF,168



Class proportions:


,percent
harmonized_label,
NV,35.00
MEL,24.81
BCC,22.79
BKL,9.61
AKIEC,6.74
DF,1.04


## Sanity check: lesion-level label consistency

A lesion should not have multiple conflicting kept labels.


In [5]:
ham_lesion_table, ham_conflicts = lesion_level_label_summary(ham_keep, "HAM keep-only")
bcn_lesion_table, bcn_conflicts = lesion_level_label_summary(bcn_keep, "BCN keep-only")



HAM keep-only: lesion-level label summary
unique lesions: 8542
lesions with >1 label: 0
No lesion-level label conflicts found.

BCN keep-only: lesion-level label summary
unique lesions: 4543
lesions with >1 label: 0
No lesion-level label conflicts found.


## Create lesion-grouped splits

### Current experiment design

**HAM**:
- used for base training
- split into `train_base` and `val_base`

**BCN**:
- used for target adaptation / evaluation
- split into `train_bcn_ft`, `val_bcn_ft`, `test_bcn`


In [6]:
train_base_df, val_base_df = grouped_split_by_lesion(
    ham_keep,
    group_col="lesion_id",
    label_col="harmonized_label",
    train_frac=0.8,
    val_frac=0.2,
    seed=SEED
)

train_bcn_ft_df, val_bcn_ft_df, test_bcn_df = grouped_split_train_val_test_by_lesion(
    bcn_keep,
    group_col="lesion_id",
    label_col="harmonized_label",
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=SEED
)

print("Split creation finished.")


Split creation finished.


## Split summaries


In [7]:
print_split_summary(train_base_df, "train_base (HAM)")
print_split_summary(val_base_df, "val_base (HAM)")

print_split_summary(train_bcn_ft_df, "train_bcn_ft (BCN)")
print_split_summary(val_bcn_ft_df, "val_bcn_ft (BCN)")
print_split_summary(test_bcn_df, "test_bcn (BCN)")



train_base (HAM) summary
rows: 9052
unique lesions: 6834
unique images: 9052

Class counts:


,count
harmonized_label,
NV,6195
MEL,1054
BCC,494
BKL,1061
AKIEC,120
DF,128



Class proportions:


,percent
harmonized_label,
NV,68.44
MEL,11.64
BCC,5.46
BKL,11.72
AKIEC,1.33
DF,1.41



val_base (HAM) summary
rows: 2258
unique lesions: 1708
unique images: 2258

Class counts:


...

,percent
harmonized_label,
NV,35.96
MEL,22.95
BCC,23.38
BKL,9.79
AKIEC,6.66
DF,1.26



test_bcn (BCN) summary
rows: 3249
unique lesions: 909
unique images: 3249

Class counts:


,count
harmonized_label,
NV,1155
MEL,819
BCC,699
BKL,317
AKIEC,231
DF,28



Class proportions:


,percent
harmonized_label,
NV,35.55
MEL,25.21
BCC,21.51
BKL,9.76
AKIEC,7.11
DF,0.86


## Leakage checks

We check that there is **no overlap** between splits for:
- `lesion_id`
- `isic_id`
- `group_id` (if present)


In [8]:
ham_lesion_overlap = check_no_overlap(train_base_df, val_base_df, "lesion_id", "train_base", "val_base")
ham_isic_overlap = check_no_overlap(train_base_df, val_base_df, "isic_id", "train_base", "val_base")

if "group_id" in train_base_df.columns and "group_id" in val_base_df.columns:
    ham_group_overlap = check_no_overlap(train_base_df, val_base_df, "group_id", "train_base", "val_base")

bcn_tv_lesion = check_no_overlap(train_bcn_ft_df, val_bcn_ft_df, "lesion_id", "train_bcn_ft", "val_bcn_ft")
bcn_tt_lesion = check_no_overlap(train_bcn_ft_df, test_bcn_df, "lesion_id", "train_bcn_ft", "test_bcn")
bcn_vt_lesion = check_no_overlap(val_bcn_ft_df, test_bcn_df, "lesion_id", "val_bcn_ft", "test_bcn")

bcn_tv_isic = check_no_overlap(train_bcn_ft_df, val_bcn_ft_df, "isic_id", "train_bcn_ft", "val_bcn_ft")
bcn_tt_isic = check_no_overlap(train_bcn_ft_df, test_bcn_df, "isic_id", "train_bcn_ft", "test_bcn")
bcn_vt_isic = check_no_overlap(val_bcn_ft_df, test_bcn_df, "isic_id", "val_bcn_ft", "test_bcn")

if "group_id" in train_bcn_ft_df.columns:
    bcn_tv_group = check_no_overlap(train_bcn_ft_df, val_bcn_ft_df, "group_id", "train_bcn_ft", "val_bcn_ft")
    bcn_tt_group = check_no_overlap(train_bcn_ft_df, test_bcn_df, "group_id", "train_bcn_ft", "test_bcn")
    bcn_vt_group = check_no_overlap(val_bcn_ft_df, test_bcn_df, "group_id", "val_bcn_ft", "test_bcn")


Overlap check for 'lesion_id' between train_base and val_base: 0
Overlap check for 'isic_id' between train_base and val_base: 0
Overlap check for 'group_id' between train_base and val_base: 0
Overlap check for 'lesion_id' between train_bcn_ft and val_bcn_ft: 0
Overlap check for 'lesion_id' between train_bcn_ft and test_bcn: 0
Overlap check for 'lesion_id' between val_bcn_ft and test_bcn: 0
Overlap check for 'isic_id' between train_bcn_ft and val_bcn_ft: 0
Overlap check for 'isic_id' between train_bcn_ft and test_bcn: 0
Overlap check for 'isic_id' between val_bcn_ft and test_bcn: 0
Overlap check for 'group_id' between train_bcn_ft and val_bcn_ft: 0
Overlap check for 'group_id' between train_bcn_ft and test_bcn: 0
Overlap check for 'group_id' between val_bcn_ft and test_bcn: 0


## Combined split overview table


In [9]:
overview_rows = []

for name, df in [
    ("train_base", train_base_df),
    ("val_base", val_base_df),
    ("train_bcn_ft", train_bcn_ft_df),
    ("val_bcn_ft", val_bcn_ft_df),
    ("test_bcn", test_bcn_df),
]:
    row = {
        "split": name,
        "n_rows": len(df),
        "n_lesions": df["lesion_id"].nunique(dropna=True),
        "n_images": df["isic_id"].nunique(dropna=True),
    }
    for cls in PRIMARY_CLASSES:
        row[f"class_{cls}"] = int((df["harmonized_label"] == cls).sum())
    overview_rows.append(row)

overview_df = pd.DataFrame(overview_rows)
display(overview_df)


,split,n_rows,n_lesions,n_images,class_NV,class_MEL,class_BCC,class_BKL,class_AKIEC,class_DF
0,train_base,9052,6834,9052,6195,1054,494,1061,120,128
1,val_base,2258,1708,2258,1541,251,128,277,29,32
2,train_bcn_ft,9625,2725,9625,3320,2436,2215,915,640,99
3,val_bcn_ft,3259,909,3259,1172,748,762,319,217,41
4,test_bcn,3249,909,3249,1155,819,699,317,231,28


## Save final split CSVs


In [10]:
save_split(train_base_df, OUT_DIR / "train_base.csv")
save_split(val_base_df, OUT_DIR / "val_base.csv")
save_split(train_bcn_ft_df, OUT_DIR / "train_bcn_ft.csv")
save_split(val_bcn_ft_df, OUT_DIR / "val_bcn_ft.csv")
save_split(test_bcn_df, OUT_DIR / "test_bcn.csv")


Saved: ../data/processed/splits/train_base.csv | shape=(9052, 32)
Saved: ../data/processed/splits/val_base.csv | shape=(2258, 32)
Saved: ../data/processed/splits/train_bcn_ft.csv | shape=(9625, 32)
Saved: ../data/processed/splits/val_bcn_ft.csv | shape=(3259, 32)
Saved: ../data/processed/splits/test_bcn.csv | shape=(3249, 32)


## Reload saved files for one last sanity check


In [11]:
train_base_check = pd.read_csv(OUT_DIR / "train_base.csv")
val_base_check = pd.read_csv(OUT_DIR / "val_base.csv")
train_bcn_ft_check = pd.read_csv(OUT_DIR / "train_bcn_ft.csv")
val_bcn_ft_check = pd.read_csv(OUT_DIR / "val_bcn_ft.csv")
test_bcn_check = pd.read_csv(OUT_DIR / "test_bcn.csv")

print("Reloaded shapes:")
print("train_base:", train_base_check.shape)
print("val_base:", val_base_check.shape)
print("train_bcn_ft:", train_bcn_ft_check.shape)
print("val_bcn_ft:", val_bcn_ft_check.shape)
print("test_bcn:", test_bcn_check.shape)


Reloaded shapes:
train_base: (9052, 32)
val_base: (2258, 32)
train_bcn_ft: (9625, 32)
val_bcn_ft: (3259, 32)
test_bcn: (3249, 32)


## What to do next

After this notebook succeeds, the next step is:

### `05_train_baseline_convnext.ipynb`

That notebook should:
1. load these split CSVs
2. build a PyTorch dataset from `image_path` and `harmonized_label`
3. initialize **ConvNeXt-Tiny** with ImageNet weights
4. train on `train_base.csv`, validate on `val_base.csv`
5. fine-tune on `train_bcn_ft.csv`, validate on `val_bcn_ft.csv`
6. evaluate on `test_bcn.csv`

Only after that should you move on to:
- Grad-CAM
- LayerCAM
- difference-CAM
- saliency separation loss
